# 🔧 Bloque 5: Feature Engineering y Evaluación

**Objetivo:** Dominar el preprocesamiento de datos y la evaluación correcta de modelos. En la práctica, este bloque define el 80% de la calidad de un modelo.

---

## 1. ¿Por qué importa tanto?

> *"Garbage in, garbage out"* — si los datos de entrada son malos, el modelo será malo sin importar su arquitectura.

Feature Engineering es el arte de transformar datos crudos en representaciones que los modelos puedan aprender mejor.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

# Dataset sintético con problemas reales
np.random.seed(42)
n = 200

df = pd.DataFrame({
    'edad':      np.random.normal(35, 10, n),
    'salario':   np.random.lognormal(10, 0.5, n),
    'ciudad':    np.random.choice(['Madrid', 'Barcelona', 'Valencia', 'Sevilla'], n),
    'educacion': np.random.choice(['Primaria', 'Secundaria', 'Universidad', 'Master'], n),
    'target':    np.random.randint(0, 2, n)
})

# Introducir valores nulos artificialmente
df.loc[np.random.choice(n, 20, replace=False), 'edad'] = np.nan
df.loc[np.random.choice(n, 10, replace=False), 'salario'] = np.nan

# Introducir outlier
df.loc[5, 'salario'] = 10_000_000

print(df.head())
print(f"\nShape: {df.shape}")
print(f"\nNulos por columna:\n{df.isnull().sum()}")

## 2. Manejo de valores nulos

In [ ]:
# Estrategias para nulos numéricos
imputer_mean   = SimpleImputer(strategy='mean')       # Reemplaza con media
imputer_median = SimpleImputer(strategy='median')     # Más robusto a outliers
imputer_const  = SimpleImputer(strategy='constant', fill_value=-1)  # Valor fijo

df_clean = df.copy()
df_clean['edad']    = imputer_median.fit_transform(df[['edad']]).ravel()
df_clean['salario'] = imputer_median.fit_transform(df[['salario']]).ravel()

print(f"Nulos tras imputación: {df_clean.isnull().sum().sum()}")

## 3. Detección y tratamiento de outliers

In [ ]:
# Método IQR (Interquartile Range)
Q1 = df_clean['salario'].quantile(0.25)
Q3 = df_clean['salario'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df_clean[(df_clean['salario'] < lower) | (df_clean['salario'] > upper)]
print(f"Outliers detectados: {len(outliers)}")
print(f"Rango válido: [{lower:,.0f}, {upper:,.0f}]")

# Opción 1: eliminar
df_no_outliers = df_clean[(df_clean['salario'] >= lower) & (df_clean['salario'] <= upper)]

# Opción 2: capping (recortar en los límites)
df_clean['salario_capped'] = df_clean['salario'].clip(lower, upper)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
df_clean['salario'].hist(bins=30, ax=axes[0], color='coral')
axes[0].set_title('Salario original (con outlier)')
df_clean['salario_capped'].hist(bins=30, ax=axes[1], color='steelblue')
axes[1].set_title('Salario tras capping')
plt.tight_layout()
plt.show()

## 4. Encoding de variables categóricas

In [ ]:
# One-Hot Encoding (para variables nominales sin orden)
ciudad_dummies = pd.get_dummies(df_clean['ciudad'], prefix='ciudad')
print("One-Hot Encoding de 'ciudad':")
print(ciudad_dummies.head())

# Label Encoding (para variables ordinales con orden)
orden_educacion = {'Primaria': 0, 'Secundaria': 1, 'Universidad': 2, 'Master': 3}
df_clean['educacion_enc'] = df_clean['educacion'].map(orden_educacion)
print(f"\nLabel Encoding de 'educacion':")
print(df_clean[['educacion', 'educacion_enc']].drop_duplicates().sort_values('educacion_enc'))

## 5. Escalado de features

In [ ]:
# StandardScaler: media=0, std=1 (para algoritmos que asumen distribución normal)
# MinMaxScaler: rango [0,1] (para redes neuronales e imágenes)

features_num = ['edad', 'salario_capped']

std_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

scaled_std    = std_scaler.fit_transform(df_clean[features_num])
scaled_minmax = minmax_scaler.fit_transform(df_clean[features_num])

print(f"StandardScaler — media: {scaled_std.mean(axis=0).round(3)}, std: {scaled_std.std(axis=0).round(3)}")
print(f"MinMaxScaler   — min: {scaled_minmax.min(axis=0).round(3)}, max: {scaled_minmax.max(axis=0).round(3)}")

## 6. Selección de features

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif

# Construir dataset final
X_final = pd.concat([
    pd.DataFrame(scaled_std, columns=['edad_scaled', 'salario_scaled']),
    ciudad_dummies.reset_index(drop=True),
    df_clean['educacion_enc'].reset_index(drop=True)
], axis=1)
y_final = df_clean['target'].reset_index(drop=True)

# Importancia de features con Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_final, y_final)

importances = pd.Series(rf.feature_importances_, index=X_final.columns)
importances.sort_values().plot(kind='barh', color='steelblue', figsize=(8, 4))
plt.title('Importancia de Features (Random Forest)')
plt.tight_layout()
plt.show()

## 7. Métricas avanzadas de evaluación

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, average_precision_score

X_tr, X_te, y_tr, y_te = train_test_split(X_final, y_final, test_size=0.2, random_state=42)
rf.fit(X_tr, y_tr)

y_proba = rf.predict_proba(X_te)[:, 1]  # Probabilidad de clase positiva

auc = roc_auc_score(y_te, y_proba)
ap  = average_precision_score(y_te, y_proba)

fpr, tpr, _ = roc_curve(y_te, y_proba)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='steelblue', linewidth=2, label=f'ROC Curve (AUC = {auc:.3f})')
plt.plot([0,1], [0,1], 'k--', alpha=0.5, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC')
plt.legend()
plt.tight_layout()
plt.show()

print(f"ROC-AUC: {auc:.3f}")
print(f"Average Precision: {ap:.3f}")

---

## ✅ Resumen del bloque

- Manejaste **valores nulos** con distintas estrategias de imputación
- Detectaste y trataste **outliers** con IQR y capping
- Aplicaste **encoding** correcto según el tipo de variable (nominal vs ordinal)
- Escalaste features con **StandardScaler y MinMaxScaler**
- Calculaste **importancia de features** y métricas avanzadas (ROC-AUC)

---

## ➡️ Siguiente paso

Continúa con el **Bloque 6: HPC y Linux** → `06_hpc_linux_workflows.ipynb`